# RNA3D — DRfold Pilot10: 33-nt first-target runtime gate (GPU)

**Next model: original DRfold, not DRfold2.** This notebook uses the *already verified* DRfold Colab setup from the benchmark repository, pinned to project commit `2946e14411b75be1621635164733d25c4d4113de` and upstream DRfold commit `9586990c79e4ca488e5f967fcc8bd5b06cd60273`. It **does not rerun the 77-nt bundled inference**. It runs one frozen pilot target (`9g4p_B`, 33 nt) to measure practical runtime first.

**Runtime: T4 GPU.** Prior official 77-nt smoke took **5669.692 s** (~94.5 min) and 1395 MiB GPU memory peak; shorter RNA may still take substantial time. Results are preserved in a new run-ID folder; **never overwrite old experiments**. Native PDBs are not used for inference. Nothing is pushed to GitHub.

Run cells **in order**. The environment setup cell inherits and executes only the original notebook's code cells through **section 14: Feature preflight**; all later bundled and custom-target inference cells are deliberately skipped. If any setup step fails, it stops.

## 1 — Mount the benchmark Drive and verify the frozen target

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, re, os, sys, subprocess, time, csv, shutil, threading
ROOT=Path('/content/drive/MyDrive/RNA3D_Benchmark')
TARGET='9g4p_B'  # shortest frozen pilot RNA, first runtime-gate trial
RUN_ID='drfold_pilot10_single_20261115_v1'
SOURCE=ROOT/'NuFold/runs/pilot10_single_20261018T203832Z/inputs'/TARGET/f'{TARGET}.fasta'
OUT_BASE=ROOT/'DRfold/runs'/RUN_ID
DEST=OUT_BASE/TARGET
assert ROOT.is_dir() and SOURCE.is_file(), f'Missing Drive benchmark FASTA: {SOURCE}'
assert not DEST.exists(), f'Immutable output already exists: {DEST}. Inspect before rerun; do not overwrite.'
raw=[x.strip() for x in SOURCE.read_text().splitlines() if x.strip()]
assert raw[0].startswith('>') and sum(x.startswith('>') for x in raw)==1
SEQ=''.join(x for x in raw if not x.startswith('>')).upper()
assert re.fullmatch('[ACGU]+',SEQ),f'Bad RNA sequence: {SEQ}'
assert len(SEQ)==33, f'Frozen pilot length mismatch ({len(SEQ)})'
def sha256(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
print('Target:',TARGET,'length:',len(SEQ))
print('Frozen FASTA:',SOURCE,'SHA256:',sha256(SOURCE))
print('Output folder (new):',DEST)
print('Native coordinates are not opened or used.')

## 2 — Run the prior verified DRfold setup, not its time-consuming bundled inference

The following cell pins **both** the benchmark launcher and official DRfold source. It executes the existing documented environment installation and feature preflight (original code cells up to and including cell 28). It can take time and download dependencies. Read the existing source notebook in the benchmark repository for the full setup steps. **Do not run an unpinned main branch.**

In [ ]:
import nbformat
PROJECT=Path('/content/rna3d-drfold-setup')
PROJECT_SHA='2946e14411b75be1621635164733d25c4d4113de'
UPSTREAM_SHA='9586990c79e4ca488e5f967fcc8bd5b06cd60273'
if not PROJECT.exists():
    subprocess.run(['git','clone','--filter=blob:none','https://github.com/ABDULSALAM589/rna3d-template-sparse-benchmark.git',str(PROJECT)],check=True)
subprocess.run(['git','-C',str(PROJECT),'checkout','--detach',PROJECT_SHA],check=True)
assert subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'],text=True).strip()==PROJECT_SHA
REPO=Path('/content/DRfold')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/leeyang/DRfold.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',UPSTREAM_SHA],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==UPSTREAM_SHA
source=nbformat.read(PROJECT/'notebooks/drfold_colab.ipynb',as_version=4)
# Reuse only the previously verified installation + feature preflight, not bundled inference.
steps=[(i,c) for i,c in enumerate(source.cells[:29]) if c.cell_type=='code']
assert [i for i,_ in steps]==list(range(2,29,2)),'Upstream setup notebook changed; inspect before running'
for i,c in steps:
    print('\n'+'='*60+'\nRunning verified legacy setup code cell',i,'/ 28',flush=True)
    result=get_ipython().run_cell(c.source)
    if result.error_before_exec or result.error_in_exec:
        raise RuntimeError(f'DRfold setup cell {i} FAILED; stop here') from (result.error_in_exec or result.error_before_exec)
print('SETUP + FEATURE PREFLIGHT PASS')
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==UPSTREAM_SHA
assert Path('/content/DRfold/DRfold.sh').is_file()

## 3 — Frozen FASTA to local staging; validate DRfold dependencies

This checks the target sequence and environment again before the expensive run. Only FASTA is passed to DRfold. PETfold/ViennaRNA are its **internal** features, not a supplied genuine MSA.

In [ ]:
LOCAL=Path('/content/drfold_pilot10_stage')/TARGET
if LOCAL.exists():
    raise FileExistsError(f'Previous local work exists: {LOCAL}. Inspect before rerunning.')
LOCAL.mkdir(parents=True)
FASTA=LOCAL/f'{TARGET}.fasta'
FASTA.write_text(f'>{TARGET}\n{SEQ}\n')
assert re.fullmatch('[ACGU]+',''.join(FASTA.read_text().splitlines()[1:]))
PDB_DIR=LOCAL/'model_output'
PYTHON_EXE=Path('/content/micromamba_root/envs/drfold/bin/python')
DRFOLD_SH=Path('/content/DRfold/DRfold.sh')
assert PYTHON_EXE.is_file() and DRFOLD_SH.is_file()
assert 'PETFOLD_REAL' in globals() and Path(PETFOLD_REAL).is_file()
assert 'VIENNA_PREFIX' in globals() and Path(VIENNA_PREFIX).is_dir()
run_env=os.environ.copy()
run_env['PATH']=f"{VIENNA_PREFIX/'bin'}:{Path(PETFOLD_REAL).parent}:"+run_env['PATH']
run_env['LD_LIBRARY_PATH']=f"{VIENNA_PREFIX/'lib'}:"+run_env.get('LD_LIBRARY_PATH','')
subprocess.run(['nvidia-smi'],check=True)
print('PREFLIGHT PASS: GPU + DRfold + ViennaRNA + PETfold + frozen target')
print('Expected output:',PDB_DIR/'DPR.pdb')

## 4 — Run one real DRfold prediction (may be slow)

Runs **one** 33-nt RNA. Stream progress every 5 minutes. Captures full stdout/stderr in a local log, monitors GPU memory, and saves completed results to Drive only after output checks pass. If Colab disconnects before completion, this run may be lost; do **not** assume success until the final Drive record exists.

In [ ]:
assert not DEST.exists(),f'Existing immutable Drive result: {DEST}'
assert not PDB_DIR.exists(),f'Local prediction exists: {PDB_DIR}'
logfile=LOCAL/'inference.log'
gpu_samples=[];stop_monitor=threading.Event()
def watch_gpu():
    while not stop_monitor.wait(10):
        try:
            q=subprocess.check_output(['nvidia-smi','--query-gpu=memory.used','--format=csv,noheader,nounits'],text=True,timeout=10).splitlines()
            if q:gpu_samples.append(int(q[0].strip()))
        except Exception:pass
monitor=threading.Thread(target=watch_gpu,daemon=True);monitor.start()
start=time.monotonic()
command=['bash',str(DRFOLD_SH),str(FASTA),str(PDB_DIR)]
print('Executing DRfold:',command,'\nLog:',logfile,flush=True)
try:
    with logfile.open('w') as fh:
        proc=subprocess.Popen(command,cwd=REPO,env=run_env,stdout=fh,stderr=subprocess.STDOUT)
        while proc.poll() is None:
            try:proc.wait(timeout=300)
            except subprocess.TimeoutExpired:
                print(f'...DRfold still running, elapsed {(time.monotonic()-start)/60:.1f} min; max observed GPU RAM {max(gpu_samples) if gpu_samples else "unknown"} MiB',flush=True)
    code=proc.returncode
finally:
    stop_monitor.set();monitor.join(timeout=2)
elapsed=round(time.monotonic()-start,3)
PDB=PDB_DIR/'DPR.pdb'
print('Exit:',code,'Runtime (min):',round(elapsed/60,2),'PDB exists:',PDB.is_file())
if code!=0 or not PDB.is_file() or PDB.stat().st_size<1000:
    print('Last 4000 log chars:',logfile.read_text(errors='replace')[-4000:]);raise RuntimeError('DRfold did not complete; Drive has not been finalized')
residues=set()
for line in PDB.read_text(errors='replace').splitlines():
    if line.startswith('ATOM  '):residues.add((line[21:22],line[22:26],line[26:27]))
assert len(residues)==len(SEQ),f'PDB residue count {len(residues)} != FASTA {len(SEQ)}. Inspect before archiving.'
record={'target_id':TARGET,'model':'DRfold','model_commit':UPSTREAM_SHA,'status':'PASS','input_mode':'sequence_with_internal_petfold_viennarna','input_length':len(SEQ),'output_residues':len(residues),'runtime_seconds':elapsed,'peak_gpu_memory_mib_observed':max(gpu_samples) if gpu_samples else None,'input_fasta_sha256':sha256(SOURCE),'prediction_sha256':sha256(PDB),'exit_code':code,'native_used_for_inference':False}
# Save to a fresh immutable Drive target directory. This writes no GitHub files.
assert not DEST.exists(),f'Refusing overwrite: {DEST}'
DEST.mkdir(parents=True,exist_ok=False)
shutil.copy2(FASTA,DEST/FASTA.name)
shutil.copy2(PDB,DEST/'DPR.pdb')
shutil.copy2(logfile,DEST/'inference.log')
(DEST/'complete.json').write_text(json.dumps(record,indent=2)+'\n')
assert sha256(DEST/'DPR.pdb')==record['prediction_sha256']
assert sha256(DEST/FASTA.name)==sha256(FASTA)
print('DRFOLD FIRST TARGET PASS: saved to',DEST)
print(json.dumps(record,indent=2))

## 5 — Read-only Drive verification

Safe to rerun after Colab reconnects. This is only a first-target gate, **not 10/10 completion**.

In [ ]:
if not (DEST/'complete.json').is_file():
    raise FileNotFoundError(f'Not finalized: {DEST}')
meta=json.loads((DEST/'complete.json').read_text())
assert meta['status']=='PASS' and meta['target_id']==TARGET and meta['model_commit']==UPSTREAM_SHA
assert sha256(DEST/'DPR.pdb')==meta['prediction_sha256']
print('READ-BACK VERIFIED: 1/10 DRfold pilot targets, runtime seconds:',meta['runtime_seconds'])
print('Next: review runtime before running 9 more targets. No inference was rerun by this verification.')